In [2]:
import pickle
from sentence_transformers import SentenceTransformer, InputExample, losses, models
from torch.utils.data import DataLoader
from sentence_transformers import util
import torch

C:\Users\ohs\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
base_path = r"C:\Users\ohs\Desktop\DataFile"

with open(base_path + r"\kb_definitions.pickle", "rb") as f:
    kb_definitions = pickle.load(f)

with open(base_path + r"\kb_synonyms_labels.pickle", "rb") as f:
    kb_synonyms_labels = pickle.load(f)

In [4]:
first_key = next(iter(kb_definitions))
print(first_key, kb_definitions[first_key])

first_key = next(iter(kb_synonyms_labels))
print(first_key, kb_synonyms_labels[first_key])

http://purl.obolibrary.org/obo/BTO_0000754 Hyaline eosinophilic concentrically-laminated inclusions found in the substantia nigra and locus ceruleus of patients with Parkinsonism and Lewy body dementia.
http://purl.obolibrary.org/obo/BTO_0000754 ['lewy body']


In [5]:
train_data = []

for entity_id, aliases in kb_synonyms_labels.items():
    definition = kb_definitions.get(entity_id, "")
    
    for alias in aliases:
        train_data.append({
            "mention": alias,
            "context": definition,
            "label": entity_id
        })

In [6]:
train_examples = [
    InputExample(texts=[item["mention"], item["context"]], label=1.0)
    for item in train_data
]

In [7]:
train_loader = DataLoader(train_examples, shuffle=True, batch_size=16)

In [8]:
transformer = models.Transformer("cambridgeltl/SapBERT-UMLS-2020AB-all-lang-from-XLMR")
pooling = models.Pooling(
    transformer.get_word_embedding_dimension(),
    pooling_mode_mean_tokens=True,
    pooling_mode_cls_token=False,
    pooling_mode_max_tokens=False
)

dense = models.Dense(
    in_features=pooling.get_sentence_embedding_dimension(),
    out_features=256, 
    activation_function=torch.nn.Tanh()
)

model = SentenceTransformer(modules=[transformer, pooling, dense])

from sentence_transformers import losses

loss = losses.MultipleNegativesRankingLoss(model)

In [9]:
model.fit(
    train_objectives=[
        (train_loader, loss)],
    epochs=3,
    warmup_steps=100
)

C:\Users\ohs\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\torch\utils\data\dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


Step,Training Loss


KeyboardInterrupt: 

In [ ]:
model = model

mention = "lewy body"
query_emb = model.encode(mention, convert_to_tensor=True)

entity_ids = list(kb_definitions.keys())
entity_texts = [kb_definitions[e] for e in entity_ids]

entity_embs = model.encode(entity_texts, convert_to_tensor=True)

scores = util.cos_sim(query_emb, entity_embs)[0]

top_k = 5
top_results = scores.topk(k=top_k)

for score, idx in zip(top_results.values, top_results.indices):
    print(entity_ids[idx], float(score))